# Tech Challenge - Fase 1
## 02 - Modelagem

A partir da base processada na EDA (`wdbc_processed.csv`), treinamos e comparamos modelos de classificação para apoiar o diagnóstico de câncer de mama.

**Métrica prioritária:** *recall* da classe maligna (1). Em um contexto de triagem clínica, deixar de identificar um tumor maligno (falso negativo) é o erro mais custoso, então otimizamos para minimizá-lo.

## 1. Carga do dataset processado

In [1]:
import pandas as pd

In [2]:
df_processed = pd.read_csv("../data/processed/wdbc_processed.csv")

In [3]:
df_processed.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 2. Separação entre features (X) e alvo (y)

In [4]:
X = df_processed.drop(columns=["diagnosis"])

In [5]:
y = df_processed["diagnosis"]

## 3. Divisão treino / teste estratificada

Usamos `stratify=y` para preservar a proporção das classes em ambos os conjuntos e `random_state=42` para reprodutibilidade.

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
# stratify=y preserva a proporção de classes em treino e teste;
# random_state fixa o resultado para garantir reprodutibilidade.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
print(X_train.shape)
print(X_test.shape)

(455, 30)
(114, 30)


Confirmando que a estratificação preservou a proporção de classes em treino e teste:

In [9]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

diagnosis
0    0.626374
1    0.373626
Name: proportion, dtype: float64
diagnosis
0    0.631579
1    0.368421
Name: proportion, dtype: float64


## 4. Padronização das features (StandardScaler)

O `StandardScaler` é ajustado **apenas no conjunto de treino** (`fit_transform`) e aplicado ao teste (`transform`). Isso evita *data leakage*: o conjunto de teste não influencia os parâmetros (média e desvio) da transformação.

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [11]:
# fit_transform SOMENTE no treino: o scaler aprende média/desvio apenas com dados de treino.
# No teste usamos transform (mesmos parâmetros), evitando data leakage.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Sanidade: após a padronização, o conjunto de treino tem média ≈ 0 e desvio ≈ 1.

In [12]:
X_train_scaled.mean()

np.float64(1.6006731959430828e-17)

In [13]:
X_train_scaled.std()

np.float64(1.0)

## 5. Modelo 1 - Regressão Logística

In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

In [15]:
y_pred[:20]

array([0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0])

Matriz de confusão (linhas = real, colunas = previsto):

In [17]:
from sklearn.metrics import confusion_matrix

# Linhas = valor real, colunas = previsto.
# A posição [1, 0] corresponde a um falso negativo - o erro mais crítico aqui.
matrix_confusion = confusion_matrix(y_test, y_pred)
print(matrix_confusion)

[[71  1]
 [ 3 39]]


Relatório de classificação completo:

In [18]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.99      0.97        72
           1       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

